# GraphRAG with Local Attribution

Keep the source chunk on every edge so each fact in the answer can be traced to the original text.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

**Extra requirements:** `pip install networkx`

Uses the shared helpers in `../common.py` (local embeddings + Ollama). Make sure `ollama serve` is running and `ollama pull llama3.2` is done.

In [ ]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

In [ ]:
import networkx as nx

G = nx.MultiDiGraph()
for i, chunk in enumerate(chunks[:15]):
    for s, r, o in extract_triples(chunk):
        G.add_edge(s, o, relation=r, source=i)          # <-- provenance lives on the edge
print(G.number_of_edges(), "attributed edges")

In [ ]:
def attributed_facts(query, k=8):
    rel = [(f"{u} {d['relation']} {v}", d["source"]) for u, v, d in G.edges(data=True)]
    store_rel = build_store([t for t, _ in rel], [{"source": s} for _, s in rel])
    return [(h["text"], h["meta"]["source"]) for h in store_rel.search(query, k)]

query = "What does the handbook say about checking answers against sources?"
facts = attributed_facts(query)
numbered = "\n".join(f"[{src}] {fact}" for fact, src in facts)
print(llm(f"Facts (each prefixed with its source chunk id):\n{numbered}\n\nQuestion: {query}\n\nAnswer using the facts and cite chunk ids like [3].", system=RAG_SYSTEM))

Verify: print the cited chunks so a reader can check the claim against the original text.

In [ ]:
for src in sorted({s for _, s in facts}):
    print(f"--- chunk {src}\n{chunks[src]}\n")